# rdepth 3차: R 스윕 (R=2/4/6) — 회복률-R 곡선

**사용법**: 런타임 유형 **L4 GPU** → **런타임 → 모두 실행** → 드라이브 허용 + 처음 한 번 `rdepth_code_v3.zip` 업로드.

- 1차 사이클의 small/loop(R=3)/large 결과(드라이브 csv)를 재사용해 R=3 점을 채웁니다
- 각 런 200M 토큰 (1차와 동일 예산·시드·데이터), 총 ~3시간/~15유닛(L4) 예상
- 끊기면 다시 '모두 실행' — 체크포인트에서 이어짐

In [1]:
!nvidia-smi -L

GPU 0: NVIDIA L4 (UUID: GPU-cfd5db0d-22b5-e132-55b8-1182a442d26d)


In [2]:
# [1] 드라이브 + 코드 v3 (재개 수정 내장판)
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
DRIVE = '/content/drive/MyDrive/rdepth_out'
os.makedirs(DRIVE, exist_ok=True)
zpath = f'{DRIVE}/rdepth_code_v3.zip'
if not os.path.exists(zpath):
    from google.colab import files
    print('rdepth_code_v3.zip 파일을 선택해 주세요:')
    up = files.upload()
    shutil.move(list(up)[0], zpath)
os.system(f'unzip -q -o {zpath} -d /content/rdepth')
%cd /content/rdepth
assert 'map_location="cpu"' in open('train.py').read(), '구버전 zip입니다 — 드라이브의 rdepth_code_v3.zip을 지우고 다시 업로드하세요'
print('코드 v3 준비 완료')

Mounted at /content/drive
/content/rdepth
코드 v3 준비 완료


In [3]:
# [2] 데이터 (드라이브 캐시 ~1분)
%cd /content/rdepth
import os
DRIVE = '/content/drive/MyDrive/rdepth_out'
os.makedirs('data', exist_ok=True)
if os.path.exists(f'{DRIVE}/train.bin'):
    print('Drive 캐시에서 복사...')
    !cp {DRIVE}/tok4096.json {DRIVE}/val.bin {DRIVE}/train.bin data/
else:
    !python prepare_data.py
    !cp data/tok4096.json data/val.bin data/train.bin {DRIVE}/
!python -m pytest tests/test_model.py -q
print('데이터+테스트 완료')

/content/rdepth
Drive 캐시에서 복사...
......                                                                   [100%]
6 passed in 7.83s
데이터+테스트 완료


In [4]:
# [3] dtype 자동 선택 (L4/A100→bf16, T4→fp16)
import torch, os
os.environ['RDEPTH_OUT'] = '/content/drive/MyDrive/rdepth_out'
cap = torch.cuda.get_device_capability()
DTYPE = 'bf16' if cap[0] >= 8 else 'fp16'
print('GPU =', torch.cuda.get_device_name(0), cap, '| dtype =', DTYPE)

GPU = NVIDIA L4 (8, 9) | dtype = bf16


In [5]:
# [4-1] R=2 (유효 깊이 6) — 1차와 동일 예산 200M
%cd /content/rdepth
!python train.py --run loop-r2 --max-tokens 200000000 --dtype {DTYPE} --micro-batch 64 --resume

/content/rdepth
run=loop-r2 unique_params=15,209,984 resident_fp16=30.4MB dev=cuda dtype=bf16
step 50/6103 loss 6.2554 102,645 tok/s
step 100/6103 loss 4.8068 105,402 tok/s
step 150/6103 loss 4.1369 106,258 tok/s
step 200/6103 loss 3.8207 106,622 tok/s
step 250/6103 loss 3.6684 106,789 tok/s
step 300/6103 loss 3.3059 106,836 tok/s
step 350/6103 loss 3.0797 106,840 tok/s
step 400/6103 loss 2.9330 106,790 tok/s
step 450/6103 loss 2.7553 106,722 tok/s
step 500/6103 loss 2.6550 106,670 tok/s
  [eval] step 500 val 2.6691
step 550/6103 loss 2.5431 100,908 tok/s
step 600/6103 loss 2.5003 101,368 tok/s
step 650/6103 loss 2.4422 101,755 tok/s
step 700/6103 loss 2.2896 102,080 tok/s
step 750/6103 loss 2.2559 102,360 tok/s
step 800/6103 loss 2.2349 102,605 tok/s
step 850/6103 loss 2.1641 102,828 tok/s
step 900/6103 loss 2.1405 103,031 tok/s
step 950/6103 loss 2.0372 103,214 tok/s
step 1000/6103 loss 2.0310 103,376 tok/s
  [eval] step 1000 val 2.1044
step 1050/6103 loss 2.1135 100,591 tok/s
step 1

In [6]:
# [4-2] R=4 (유효 깊이 10)
%cd /content/rdepth
!python train.py --run loop-r4 --max-tokens 200000000 --dtype {DTYPE} --micro-batch 64 --resume

/content/rdepth
run=loop-r4 unique_params=15,211,008 resident_fp16=30.4MB dev=cuda dtype=bf16
step 50/6103 loss 6.3416 65,920 tok/s
step 100/6103 loss 4.8691 66,376 tok/s
step 150/6103 loss 4.1837 66,578 tok/s
step 200/6103 loss 3.8649 66,692 tok/s
step 250/6103 loss 3.7065 66,732 tok/s
step 300/6103 loss 3.3544 66,759 tok/s
step 350/6103 loss 3.1178 66,786 tok/s
step 400/6103 loss 2.9568 66,813 tok/s
step 450/6103 loss 2.7751 66,822 tok/s
step 500/6103 loss 2.6615 66,829 tok/s
  [eval] step 500 val 2.6886
step 550/6103 loss 2.5460 63,312 tok/s
step 600/6103 loss 2.5074 63,596 tok/s
step 650/6103 loss 2.4527 63,838 tok/s
step 700/6103 loss 2.2906 64,049 tok/s
step 750/6103 loss 2.2656 64,235 tok/s
step 800/6103 loss 2.2349 64,395 tok/s
step 850/6103 loss 2.1644 64,538 tok/s
step 900/6103 loss 2.1387 64,666 tok/s
step 950/6103 loss 2.0240 64,781 tok/s
step 1000/6103 loss 2.0202 64,885 tok/s
  [eval] step 1000 val 2.0976
step 1050/6103 loss 2.1009 63,182 tok/s
step 1100/6103 loss 2.0601 

In [7]:
# [4-3] R=6 (유효 깊이 14)
%cd /content/rdepth
!python train.py --run loop-r6 --max-tokens 200000000 --dtype {DTYPE} --micro-batch 64 --resume

/content/rdepth
run=loop-r6 unique_params=15,212,032 resident_fp16=30.4MB dev=cuda dtype=bf16
step 50/6103 loss 6.3733 48,298 tok/s
step 100/6103 loss 5.0059 48,604 tok/s
step 150/6103 loss 4.2353 48,715 tok/s
step 200/6103 loss 3.8975 48,752 tok/s
step 250/6103 loss 3.7751 48,781 tok/s
step 300/6103 loss 3.3958 48,803 tok/s
step 350/6103 loss 3.1433 48,819 tok/s
step 400/6103 loss 2.9718 48,829 tok/s
step 450/6103 loss 2.7872 48,838 tok/s
step 500/6103 loss 2.6849 48,845 tok/s
  [eval] step 500 val 2.7047
step 550/6103 loss 2.5541 46,297 tok/s
step 600/6103 loss 2.5244 46,504 tok/s
step 650/6103 loss 2.4674 46,680 tok/s
step 700/6103 loss 2.3034 46,832 tok/s
step 750/6103 loss 2.2725 46,964 tok/s
step 800/6103 loss 2.2439 47,080 tok/s
step 850/6103 loss 2.1709 47,184 tok/s
step 900/6103 loss 2.1469 47,276 tok/s
step 950/6103 loss 2.0326 47,359 tok/s
step 1000/6103 loss 2.0310 47,434 tok/s
  [eval] step 1000 val 2.0993
step 1050/6103 loss 2.1156 46,200 tok/s
step 1100/6103 loss 2.0647 

In [8]:
# [5] 회복률-R 곡선 (1차 결과 재사용: small, loop=R3, large)
import csv, os
out = '/content/drive/MyDrive/rdepth_out'
def best(r):
    p = f'{out}/logs/{r}.csv'
    if not os.path.exists(p): return None
    with open(p) as f: data = list(csv.DictReader(f))
    return min(float(d['val_loss']) for d in data) if data else None
s, g = best('small'), best('large')
print(f'기준선 small(R1,깊이4)={s}  상한 large(깊이8)={g}')
for name, R, depth in [('loop-r2', 2, 6), ('loop', 3, 8), ('loop-r4', 4, 10), ('loop-r6', 6, 14)]:
    v = best(name)
    if v is None or s is None or g is None:
        print(f'R={R}: (미완료)')
    else:
        rec = (s - v) / (s - g) * 100
        print(f'R={R} (깊이{depth}): val {v:.4f}  회복률 {rec:.1f}%  {"← large 추월!" if v < g else ""}')

기준선 small(R1,깊이4)=1.595  상한 large(깊이8)=1.5045
R=2 (깊이6): val 1.5700  회복률 27.6%  
R=3 (깊이8): val 1.5573  회복률 41.7%  
R=4 (깊이10): val 1.5511  회복률 48.5%  
R=6 (깊이14): val 1.5482  회복률 51.7%  
